# Topomap plots — Task Difference Simple−Complex (Load paper, active only)

Plots topographic maps of the within-subject Simple−Complex task difference.
Reads from `outputs/grand_averages/load/taskdiff/{TRIALS}/` pkls
(`TRIALS` = `'correct'` (primary) or `'alltrials'`, set in the config cell; added to figure names).

Confirmed plots:
- Target N2: 180–230ms
- Novel N2:  150–220ms

Batch configs live in `config_load.BATCH_TOPO_TASKDIFF`.

Last edit: 31.10.2026 @anguyen

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import mne
from mne import grand_average
from mne.defaults import DEFAULTS
from mne.viz import set_browser_backend

def find_repo_root(marker='core') -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root() / 'core'))
from config_load import BATCH_TOPO_TASKDIFF
from participants import get_participants
from utils import load_pkl, show_latest_pkls, window_average_evoked, get_topomap_image

%matplotlib qt
set_browser_backend('qt')

plt.rcParams.update({
    'font.size':        12,
    'axes.titlesize':   13,
    'axes.labelsize':   12,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  10,
    'figure.titlesize': 16,
})

In [ ]:
# ==============================================================================
# Configuration — load paper, active only
# ==============================================================================

PAPER    = 'load'
PARADIGM = 'active'
MEASURE  = 'raw'
LOCK     = 'stim'
TRIALS   = 'correct'   # 'correct' (primary) | 'alltrials'

DEVIANT_LABEL = 'target'
DATE_STR      = datetime.today().strftime('%Y%m%d')

if sys.platform == 'win32':
    root = Path('L:/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2')
else:
    root = Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2')

assert TRIALS in ('correct', 'alltrials'), f'Unknown TRIALS: {TRIALS}'
input_path = root / 'outputs' / 'grand_averages' / PAPER / 'taskdiff' / TRIALS
root_img   = root / 'outputs' / 'img' / PAPER / 'taskdiff'
root_img.mkdir(parents=True, exist_ok=True)
(root_img / MEASURE).mkdir(parents=True, exist_ok=True)

n          = len(get_participants(PAPER, PARADIGM))
CONDITIONS = {'Simple\u2212Complex': 'taskdiff'}

event_types    = ['standard', DEVIANT_LABEL, 'novel']
stim_label_map = {
    'standard':    'Standard (Simple\u2212Complex)',
    DEVIANT_LABEL: f'{DEVIANT_LABEL.capitalize()} (Simple\u2212Complex)',
    'novel':       'Novel (Simple\u2212Complex)',
}


def latest_date(folder: Path, pattern: str) -> str:
    '''Most recent date prefix among pkl files matching `pattern` in `folder`.'''
    files = sorted(folder.glob(pattern))
    assert files, f'No pkl matching {pattern} in {folder}'
    dates = sorted({f.name.split('_')[0] for f in files})
    print(f'Available dates: {dates}')
    return dates[-1]

DATE_SOURCE = latest_date(input_path, f'*_{PARADIGM}_*p_taskdiff_stim_raw.pkl')   # or pin, e.g. '20261031'
print(f'DATE_SOURCE: {DATE_SOURCE} | Trials: {TRIALS}')

print(f'Input : {input_path}')
print(f'Output: {root_img}')

In [ ]:
# ==============================================================================
# Load pkl and compute grand averages
# ==============================================================================

fname       = f'{DATE_SOURCE}_{PARADIGM}_{n}p_taskdiff_stim_raw.pkl'
raw_evokeds = load_pkl(input_path / fname)
print(f'Loaded <- {fname}')
print(f'Keys: {list(raw_evokeds.keys())}')

cond_key = 'Simple\u2212Complex'
ga: dict[str, dict] = {cond_key: {}}
for stim in event_types:
    if stim in raw_evokeds:
        ga[cond_key][stim] = grand_average(
            [e.copy().pick('eeg') for e in raw_evokeds[stim]]
        )
    else:
        print(f"Warning: '{stim}' not found in pkl")

print(f'\nGrand averages: {list(ga[cond_key].keys())}')

In [ ]:
def plot_topomap_grid(
    plot_windows,
    fname_suffix,
    manual_scale=None,
    scalings_override=None,
    fig_width=4,
    save=True,
):
    col_labels = list(CONDITIONS.keys())
    n_cols     = len(col_labels)
    n_rows     = len(plot_windows)

    row_labels = [
        f"{stim_label_map.get(stim, stim.capitalize())}\n{a}\u2013{b} ms"
        for stim, a, b in plot_windows
    ]

    # global colour scale
    all_imgs = []
    for stim, a, b in plot_windows:
        for label in col_labels:
            if stim in ga[label]:
                ev = window_average_evoked(ga[label][stim], a, b)
                all_imgs.append(get_topomap_image(ev))

    vmin = min(img.get_clim()[0] for img in all_imgs)
    vmax = max(img.get_clim()[1] for img in all_imgs)
    print(f'Auto scale: vmin={vmin:.4f}, vmax={vmax:.4f}')
    if manual_scale is not None:
        vmin, vmax = manual_scale
        print(f'Manual scale applied: vmin={vmin:.4f}, vmax={vmax:.4f}')

    last_ev = window_average_evoked(
        ga[col_labels[0]][plot_windows[-1][0]],
        plot_windows[-1][1], plot_windows[-1][2]
    )
    ch_type = mne.channel_type(last_ev.info, 0)
    unit    = DEFAULTS['units'][ch_type]

    fig_height = n_rows * 2.5 + 1
    fig, axes  = plt.subplots(n_rows, n_cols, figsize=(fig_width * n_cols / 2, fig_height))
    axes       = np.atleast_2d(axes)

    images      = []
    topo_kwargs = dict(colorbar=False, show=False, time_format='')
    if scalings_override is not None:
        topo_kwargs['scalings'] = scalings_override

    for row, (stim, a, b) in enumerate(plot_windows):
        for col, label in enumerate(col_labels):
            ax = axes[row, col]
            if stim not in ga[label]:
                print(f"Warning: stim='{stim}' not in ga['{label}'] — skipping.")
                continue

            ev = window_average_evoked(ga[label][stim], a, b)
            ev.plot_topomap(times=ev.times, axes=ax, **topo_kwargs)

            img = ax.images[0]
            img.set_clim(vmin, vmax)
            images.append(img)

        axes[row, 0].set_ylabel(row_labels[row], fontsize=11, labelpad=12)

    for col, label in enumerate(col_labels):
        axes[0, col].set_title(label, fontsize=12)

    bottom_margin = max(0.04, 0.12 / n_rows)
    fig.subplots_adjust(bottom=bottom_margin, hspace=0.0)

    cbar_height = 0.08 / n_rows
    cbar_ax     = fig.add_axes([0.2, bottom_margin * 0.4, 0.6, cbar_height])
    cbar        = fig.colorbar(images[0], cax=cbar_ax, orientation='horizontal')
    cbar.set_label(f'Amplitude ({unit})')

    fig.suptitle(f'[{PARADIGM.capitalize()}] Task Diff (Simple\u2212Complex)')

    if save:
        n_p   = list(ga[col_labels[0]].values())[0].nave
        fname = (
            root_img / MEASURE
            / f'{PARADIGM}_{n_p}p_{PAPER}_{MEASURE}_taskdiff_topo'
              f'_{fname_suffix}_mean_{TRIALS}_{DATE_STR}.svg'
        )
        plt.savefig(fname, dpi=300, bbox_inches='tight')
        print(f'Saved -> {fname}')

    plt.show()
    return fig, axes

print('plot_topomap_grid() ready.')

## Batch run

Runs all pre-configured topomap grids. Edit configs in `config_load.BATCH_TOPO_TASKDIFF`.

In [ ]:
batch_configs = BATCH_TOPO_TASKDIFF.get((PARADIGM, MEASURE), [])

if batch_configs:
    print(f'Running {len(batch_configs)} topo grids [taskdiff]...\n')
    for cfg in batch_configs:
        plot_topomap_grid(**cfg)
else:
    print('No BATCH_TOPO_TASKDIFF configs found — add entries to config_load.BATCH_TOPO_TASKDIFF.')

## Live / custom plots

In [ ]:
# plot_topomap_grid(
#     plot_windows = [('target', 180, 230)],
#     fname_suffix = 'targetN2',
#     manual_scale = None,
#     save         = True,
# )